In [2]:
import os
# Fixer le répertoire de travail sur la racine de ton projet
os.chdir('/Users/7maksacod/Documents/ISM/Master_2/Memoire_big_data_ia')

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
import lightgbm as lgb

sns.set_theme(style="whitegrid")
plt.rcParams.update({'font.size': 11, 'figure.titlesize': 14})

print("📥 Chargement des données et calcul de l'interprétabilité (LightGBM)...")

# 1. Chargement du dataset Gold
df = pd.read_csv('data/processed/dataset_dakar_gold.csv', sep=';')
df.columns = df.columns.str.strip().str.lower()

# Suppression des colonnes parasites ou doublons
if 'url' in df.columns:
    df_avec_url = df[df['url'].notna()].drop_duplicates(subset=['url'], keep='first')
    df_sans_url = df[df['url'].isna()]
    if 'description_complete' in df_sans_url.columns:
        df_sans_url = df_sans_url.drop_duplicates(subset=['description_complete'], keep='first')
    df = pd.concat([df_avec_url, df_sans_url], ignore_index=True)
elif 'description_complete' in df.columns:
    df = df.drop_duplicates(subset=['description_complete'], keep='first')

df = df[(df['prix_fcfa'] >= 15000) & (df['prix_fcfa'] <= 10000000)].copy()
df['quartier'] = df['quartier'].fillna('Inconnu')
df['description_complete'] = df['description_complete'].fillna('')

# 2. Définition des caractéristiques (sans BRT)
target_col = 'prix_fcfa'
num_features = [
    'nb_pieces', 'nb_salles_bain', 'ratio_bains_chambres',
    'dist_plateau_km', 'dist_vdn_km', 'dist_mer_km',
    'has_groupe', 'has_reserve_eau', 'is_studio', 'is_villa',
    'has_piscine', 'has_ascenseur', 'has_parking', 'has_securite',
    'has_clim', 'has_vue_mer', 'is_meuble', 'has_cuisine_equipee',
    'has_balcon', 'is_agence_pro', 'longueur_description'
]
num_features = [c for c in num_features if c in df.columns]
cat_features = ['quartier']
text_feature = 'description_complete'

X = df[num_features + cat_features + [text_feature]]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Préparateur et modèle LightGBM
FRENCH_STOP_WORDS = ['de', 'la', 'le', 'et', 'à', 'en', 'un', 'une', 'du', 'des', 'les', 'pour', 'dans', 'avec', 'sur', 'par', 'au', 'aux', 'est', 'sont', 'pas', 'plus', 'aussi', 'ce', 'cette', 'ces', 'se', 'sa', 'son', 'ses', 'ne', 'ou', 'très']

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='Inconnu')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_features),
    ('text', TfidfVectorizer(max_features=120, stop_words=FRENCH_STOP_WORDS, min_df=2), text_feature)
])

lgb_model = lgb.LGBMRegressor(n_estimators=350, learning_rate=0.03, num_leaves=31, random_state=42, verbose=-1)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', TransformedTargetRegressor(
        regressor=lgb_model,
        func=np.log1p,
        inverse_func=np.expm1
    ))
])

pipeline.fit(X_train, y_train)

# 4. Extraction des noms de caractéristiques transformées
preproc_fitted = pipeline.named_steps['preprocessor']
cat_names = list(preproc_fitted.named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(cat_features))
text_names = list(preproc_fitted.named_transformers_['text'].get_feature_names_out())
all_feature_names = num_features + cat_names + text_names

importances = pipeline.named_steps['regressor'].regressor_.feature_importances_
df_imp = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
df_imp['Importance_Pct'] = (df_imp['Importance'] / df_imp['Importance'].sum()) * 100

df_imp['Feature_Clean'] = df_imp['Feature'].apply(
    lambda x: x.replace('quartier_', 'Quartier: ').replace('num__', '').replace('_', ' ').title()
)
df_top_imp = df_imp.sort_values(by='Importance_Pct', ascending=False).head(15).sort_values(by='Importance_Pct', ascending=True)

# 5. Graphique 1 : Top 15 Feature Importances (LightGBM)
fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(df_top_imp['Feature_Clean'], df_top_imp['Importance_Pct'], color='#2bc0e4')
ax.set_title('Figure 1 : Top 15 des Variables les plus Influentes (LightGBM Champion)', fontweight='bold')
ax.set_xlabel('Poids dans les Décisions du Modèle (%)')

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.2, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('eda_interpretability_feature_importance.png', dpi=300)
plt.close(fig)

# 6. Graphique 2 : Alignement Réel vs Prédit
preds_brut = pipeline.predict(X_test)

fig, ax = plt.subplots(figsize=(9, 6))
sns.scatterplot(x=y_test / 1e3, y=preds_brut / 1e3, alpha=0.5, color='#2c3e50', ax=ax)
ax.plot([0, 3500], [0, 3500], 'r--', lw=2, label='Prédiction Parfaite (y = x)')
ax.set_title('Figure 2 : Alignement des Prix Réels vs Prix Prédits (LightGBM)', fontweight='bold')
ax.set_xlabel('Loyer Réel (k FCFA)')
ax.set_ylabel('Loyer Prédit (k FCFA)')
ax.set_xlim(0, 3600)
ax.set_ylim(0, 3600)
ax.legend()
plt.tight_layout()
plt.savefig('eda_interpretability_residuals.png', dpi=300)
plt.close(fig)

print("🚀 ANALYSE D'INTERPRÉTABILITÉ TERMINÉE ET GRAPHIQUES ENREGISTRÉS !")

📥 Chargement des données et calcul de l'interprétabilité (LightGBM)...
🚀 ANALYSE D'INTERPRÉTABILITÉ TERMINÉE ET GRAPHIQUES ENREGISTRÉS !
